In [1]:
import pandas as pd
import numpy as np
import os
import re
import psycopg


In [2]:
data = pd.read_csv("/Users/anshumaansoni/PycharmProjects/chrononexia/db/scripts/OC data (Responses) - Form Responses 1.csv")
display(data)


,Timestamp,Full Name,Your Position in Symbitech,Academic Year,"A comment you may want to give, related to the event. ( Like a yearbook comment, will be used on website and insta posts. )","Describe yourself in about 30 words, and why you're a great fit in the symbitech OC",Image
0,7/22/2026 20:19:39,Aarushi Sathyanarayanan,Fest Head,Third Year,We made it. Don't ask how. (OR) Every great id...,"Part planner, part problem-solver, full-time c...",5114
1,7/22/2026 22:55:28,Anirudh Singh,Design Head,Third Year,Glad to be a part of it!,"Creative, reliable, and proactive, I enjoy wor...",5511
2,7/22/2026 22:26:59,Anushka Gupta,Co head hospitality,Second Year,"To all the people, the chaos, and the memo...","I believe I am a great fit as I’m organized, d...",4449
3,7/23/2026 12:14:29,Arush Kapur,Fest Head,Third Year,Ctrl + Z doesn’t work in life. I checked !!,"I lead with initiative, stay calm under pressu...",5113
4,7/22/2026 11:40:44,Atharva Somawanshi,Executive Hospitality,Third Year,NaN,I can do this task as i was the executive for ...,Executive_Hospitality_Atharva Somawanshi
5,7/22/2026 11:46:14,Eklavya Singh Shekhawat,Executive member Logistics,Third Year,"Where ideas spark, technology inspires, and me...",Proactive team player who enjoys taking respon...,5140
6,7/22/2026 12:44:04,Ishaan Chand,Co Head,Second Year,Sleep is temporary. SymbiTech screenshots are ...,"I'm Ishaan, a student of AIML, who thrives on ...",5570
7,7/22/2026 22:12:17,Kavish Nag,Documentation Executive,Third Year,The best stories are yet to be written. Excite...,"I am someone who likes to stay involved, help ...",5122
8,7/22/2026 12:52:36,Kavya Prashant Ahire,Event Co-Head,Second Year,"Came for the tech, stayed for the chaos, and s...","I’m a people person who loves taking charge, b...",5552
9,7/27/2026 12:48:54,Kush Mittal,Design Head,Second Year,"Every late night, every challenge, every l...",Im my own toughest critic constantly refining ...,NaN


In [3]:
# Clean up unnecessary timestamp column if present
if "Timestamp" in data.columns:
    data = data.drop(columns=["Timestamp"])


## Processing Image references and linking to actual photoshoot files


In [4]:
photoshoot_dir = "/Users/anshumaansoni/PycharmProjects/chrononexia/src/assets/Photoshoot"
list_dir = os.listdir(photoshoot_dir)

# Build matching lookup table for photoshoot filenames
file_map = {}
for f in list_dir:
    if f.startswith("."):
        continue
    base, ext = os.path.splitext(f)
    file_map[f.lower()] = f
    file_map[base.lower()] = f
    if base.lower().startswith("img_"):
        file_map[base[4:].lower()] = f

def resolve_image_path(raw_val):
    if pd.isna(raw_val):
        return None
    val_str = str(raw_val).strip()
    # Direct lookup
    if val_str.lower() in file_map:
        return f"src/assets/Photoshoot/{file_map[val_str.lower()]}"
    clean_val = val_str.replace("IMG_", "").lower()
    if clean_val in file_map:
        return f"src/assets/Photoshoot/{file_map[clean_val]}"
    # Partial substring lookup
    for key, fname in file_map.items():
        if clean_val == key or (len(clean_val) > 3 and clean_val in key):
            return f"src/assets/Photoshoot/{fname}"
    return None

data["Image_Path"] = data["Image"].apply(resolve_image_path)
display(data[["Full Name", "Image", "Image_Path"]])


,Full Name,Image,Image_Path
0,Aarushi Sathyanarayanan,5114,src/assets/Photoshoot/5114.JPG
1,Anirudh Singh,5511,src/assets/Photoshoot/5511.JPG
2,Anushka Gupta,4449,NaN
3,Arush Kapur,5113,src/assets/Photoshoot/5113.JPG
4,Atharva Somawanshi,Executive_Hospitality_Atharva Somawanshi,src/assets/Photoshoot/Executive_Hospitality_At...
5,Eklavya Singh Shekhawat,5140,src/assets/Photoshoot/5140.JPG
6,Ishaan Chand,5570,src/assets/Photoshoot/5570.JPG
7,Kavish Nag,5122,src/assets/Photoshoot/5122.JPG
8,Kavya Prashant Ahire,5552,src/assets/Photoshoot/5552.JPG
9,Kush Mittal,NaN,NaN


## Extracting Fest Heads, Executives, and Heads & Co-Heads


In [5]:
col_name = "Full Name"
col_pos = "Your Position in Symbitech"
col_year = "Academic Year"
col_comment = "A comment you may want to give, related to the event. ( Like a yearbook comment, will be used on website and insta posts. )"
col_desc = "Describe yourself in about 30 words, and why you're a great fit in the symbitech OC"
col_img = "Image_Path"

# 1. Fest Heads
fest_heads_df = data[data[col_pos].astype(str).str.strip().str.lower() == "fest head"].copy().reset_index(drop=True)

# 2. Executives
executives_df = data[data[col_pos].astype(str).str.contains("executive", case=False, na=False)].copy().reset_index(drop=True)

# 3. Heads & Co-Heads (excluding Fest Head & Executives)
exclude_mask = data[col_pos].astype(str).str.strip().str.lower().eq("fest head") | data[col_pos].astype(str).str.contains("executive", case=False, na=False)
heads_coheads_df = data[~exclude_mask].copy().reset_index(drop=True)

print(f"Fest Heads count: {len(fest_heads_df)}")
print(f"Executives count: {len(executives_df)}")
print(f"Heads & Co-Heads count: {len(heads_coheads_df)}")


Fest Heads count: 3
Executives count: 8
Heads & Co-Heads count: 24


## Push Data directly to PostgreSQL with Image Paths


In [6]:
# Connection details for PostgreSQL database
DB_CONFIG = {
    "dbname": "chrononexia",
    "user": "postgres",
    "password": "",
    "host": "localhost",
    "port": 5432
}

def push_df_to_postgres(df, table_name, db_config):
    insert_query = f"""
        INSERT INTO {table_name} (name, position, academic_year, comment, description, image_url)
        VALUES (%s, %s, %s, %s, %s, %s)
        ON CONFLICT (name) DO UPDATE SET
            position = EXCLUDED.position,
            academic_year = EXCLUDED.academic_year,
            comment = EXCLUDED.comment,
            description = EXCLUDED.description,
            image_url = EXCLUDED.image_url;
    """
    
    records = []
    for _, row in df.iterrows():
        name = str(row[col_name]).strip() if pd.notna(row[col_name]) else None
        pos = str(row[col_pos]).strip() if pd.notna(row[col_pos]) else None
        year = str(row[col_year]).strip() if pd.notna(row[col_year]) else None
        comment = str(row[col_comment]).strip() if pd.notna(row[col_comment]) else None
        desc = str(row[col_desc]).strip() if pd.notna(row[col_desc]) else None
        img = str(row[col_img]).strip() if pd.notna(row[col_img]) and row[col_img] is not None else None
        records.append((name, pos, year, comment, desc, img))
    
    with psycopg.connect(**db_config) as conn:
        with conn.cursor() as cur:
            cur.executemany(insert_query, records)
        conn.commit()
    print(f"Successfully pushed {len(records)} records into PostgreSQL table '{table_name}'")

# Execute database push
push_df_to_postgres(fest_heads_df, "fest_heads", DB_CONFIG)
push_df_to_postgres(executives_df, "executives", DB_CONFIG)
push_df_to_postgres(heads_coheads_df, "heads_and_coheads", DB_CONFIG)


Successfully pushed 3 records into PostgreSQL table 'fest_heads'
Successfully pushed 8 records into PostgreSQL table 'executives'
Successfully pushed 24 records into PostgreSQL table 'heads_and_coheads'
